In [1]:
!pip install -q nest-asyncio mcp langchain-openai langchain-mcp-adapters langgraph langchain-ollama gradio tabulate soundfile librosa seaborn
!pip install torch==2.2.2 torchvision==0.17.2 torchaudio==2.2.2 --index-url https://download.pytorch.org/whl/cu118

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 50.7 MB/s eta 0:00:00
Looking in indexes: https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.1/819.1 MB 2.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 122.3 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 103.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 82.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 51.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 89.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.5/728.5 MB 2.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 3.2 MB/s eta 0:00:0000:0100:01
  

In [2]:
!mkdir -p ARL
base="https://github.com/James-Tiny-Tjib/ARL/raw/main/ARL"
!curl -L -o "ARL/Airport_Noise_Dataset.zip" "$base/Airport%20Noise%20Dataset-20260603T141901Z-3-001.zip"
!curl -L -o "ARL/DroneAudioDataset.zip" "$base/DroneAudioDataset.zip"
!curl -L -o "ARL/best_cnn.pt" "$base/best_cnn.pt"
!curl -L -o "ARL/drone_demo_dataset.zip" "$base/drone_demo_dataset.zip"
!curl -L -o "ARL/drone_multi_classifier.pt" "$base/drone_multi_classifier.pt"
!curl -L -o "ARL/resnet50_drone_weights.pth" "$base/resnet50_drone_weights.pth"
!curl -L -o "ARL/sensor_client.pt" "$base/sensor_client.pt"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  304k    0  304k    0     0   716k      0 --:--:-- --:--:-- --:--:--  716k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  286M  100  286M    0     0  43.5M      0  0:00:06  0:00:06 --:--:-- 55.1M
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  394k  100  394k    0     0   589k      0 --:--:-- --:--:-- --:--:-- 3134k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   To

In [3]:
%%writefile model.py
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as tv_models
import random

EMBEDDING_DIM = 256
NUM_CLASSES   = 2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BASE_DIR           = "./ARL"
RF_WEIGHTS_PATH    = f"{BASE_DIR}/sensor_client.pt"
AUDIO_WEIGHTS_PATH = f"{BASE_DIR}/drone_multi_classifier.pt"
VIDEO_WEIGHTS_PATH = f"{BASE_DIR}/resnet50_drone_weights.pth"


# ── Backbone architectures ────────────────────────────────────────────────────

class IQCNN(nn.Module):
    def __init__(self, num_classes: int = 2):
        super().__init__()
        self.layer_dims = []
        self.layers = nn.ModuleList()
        self.layers.append(nn.Conv1d(2,  8,  kernel_size=7, padding=3, bias=False))
        self.layers.append(nn.Conv1d(8,  16, kernel_size=7, padding=3, bias=False))
        self.layers.append(nn.Conv1d(16, 32, kernel_size=7, padding=3, bias=False))
        self.layers.append(nn.Conv1d(32, 64, kernel_size=7, padding=3, bias=False))
        self.conv_num = len(self.layers)
        self.layers.append(nn.Linear(64,  256, bias=False))          # index 4, intercept here
        self.layers.append(nn.Linear(256, num_classes, bias=False))  # index 5, head
        self.global_avg_pool = nn.AdaptiveAvgPool1d(1)

    def forward(self, x):
        for i in range(self.conv_num):
            x = F.relu(self.layers[i](x))
        x = self.global_avg_pool(x).squeeze(-1)
        for i in range(self.conv_num, len(self.layers) - 1):
            x = F.relu(self.layers[i](x))
        return self.layers[-1](x)


class DroneCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(1,  32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(),
        )
        self.fc_layers = nn.Sequential(
            nn.Linear(64 * 32 * 11, 128), nn.ReLU(), nn.Dropout(0.3),  # fc[0..2], intercept here
            nn.Linear(128, 3),                                           # fc[3], head
        )

    def forward(self, x):
        return self.fc_layers(self.conv_layers(x))


def _build_resnet50(num_classes: int = 1) -> nn.Module:
    model = tv_models.resnet50(weights=None)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


# ── Weight loaders ────────────────────────────────────────────────────────────
def build_model(num_classes=2,model_name='IQCNN'):
  return IQCNN(num_classes=num_classes)#IQCNN(num_classes=11)
    
def load_rf_model_pt(path=RF_WEIGHTS_PATH):
    rf_weights = torch.load(path, map_location=DEVICE)
    model = build_model(num_classes=2)
    model.load_state_dict(rf_weights)
    print(f"Loaded RF model from {path}")
    return model

def load_drone_cnn(path: str = AUDIO_WEIGHTS_PATH) -> DroneCNN:
    m = DroneCNN()
    m.load_state_dict(torch.load(path, map_location="cpu"))
    return m.eval()

def load_resnet50(path: str = VIDEO_WEIGHTS_PATH) -> nn.Module:
    m = _build_resnet50(num_classes=1)
    ckpt = torch.load(path, map_location="cpu")
    m.load_state_dict(ckpt.get("model_state_dict", ckpt) if isinstance(ckpt, dict) else ckpt)
    return m.eval()


# ── Frozen encoders ───────────────────────────────────────────────────────────

class FrozenRFEncoder(nn.Module):
    def __init__(self, weights_path: str = RF_WEIGHTS_PATH):
        super().__init__()
        self.backbone = load_rf_model_pt(weights_path)
        for p in self.backbone.parameters():
            p.requires_grad = False
        self.projector = nn.Sequential(
            nn.Linear(256, EMBEDDING_DIM),
            nn.BatchNorm1d(EMBEDDING_DIM),
            nn.ReLU(inplace=True),
            nn.Linear(EMBEDDING_DIM, EMBEDDING_DIM),
        )

    def forward(self, x):
        with torch.no_grad():
            for i in range(self.backbone.conv_num):
                x = F.relu(self.backbone.layers[i](x))
            x = self.backbone.global_avg_pool(x).squeeze(-1)
            for i in range(self.backbone.conv_num, len(self.backbone.layers) - 1):
                x = F.relu(self.backbone.layers[i](x))          # [B, 256]
        return self.projector(x).unsqueeze(1)                    # [B, 1, 256]


class FrozenAudioEncoder(nn.Module):
    def __init__(self, weights_path: str = AUDIO_WEIGHTS_PATH):
        super().__init__()
        self.backbone = load_drone_cnn(weights_path)
        for p in self.backbone.parameters():
            p.requires_grad = False
        self.projector = nn.Sequential(
            nn.Linear(128, EMBEDDING_DIM),
            nn.BatchNorm1d(EMBEDDING_DIM),
            nn.ReLU(inplace=True),
            nn.Linear(EMBEDDING_DIM, EMBEDDING_DIM),
        )

    def forward(self, x):
        with torch.no_grad():
            x = self.backbone.conv_layers(x)
            x = self.backbone.fc_layers[0](x)   # Linear → [B, 128]
            x = self.backbone.fc_layers[1](x)   # ReLU
            x = self.backbone.fc_layers[2](x)   # Dropout (no-op in eval)
        return self.projector(x).unsqueeze(1)   # [B, 1, 256]


class FrozenVideoEncoder(nn.Module):
    def __init__(self, weights_path: str = VIDEO_WEIGHTS_PATH):
        super().__init__()
        self.backbone = load_resnet50(weights_path)
        for p in self.backbone.parameters():
            p.requires_grad = False
        self.feature_extractor = nn.Sequential(
            self.backbone.conv1,  self.backbone.bn1,
            self.backbone.relu,   self.backbone.maxpool,
            self.backbone.layer1, self.backbone.layer2,
            self.backbone.layer3, self.backbone.layer4,
            self.backbone.avgpool,                       # [B, 2048, 1, 1]
        )
        self.projector = nn.Sequential(
            nn.Linear(2048, EMBEDDING_DIM),
            nn.BatchNorm1d(EMBEDDING_DIM),
            nn.ReLU(inplace=True),
            nn.Linear(EMBEDDING_DIM, EMBEDDING_DIM),
        )

    def forward(self, x):
        with torch.no_grad():
            hidden = self.feature_extractor(x).flatten(1)  # [B, 2048]
        return self.projector(hidden).unsqueeze(1)          # [B, 1, 256]


# ── Multi-modal projection network ────────────────────────────────────────────

class MultiModalProjectionNetwork(nn.Module):
    def __init__(
        self,
        rf_weights_path:    str = RF_WEIGHTS_PATH,
        audio_weights_path: str = AUDIO_WEIGHTS_PATH,
        video_weights_path: str = VIDEO_WEIGHTS_PATH,
    ):
        super().__init__()
        self.rf_encoder    = FrozenRFEncoder(rf_weights_path)
        self.audio_encoder = FrozenAudioEncoder(audio_weights_path)
        self.video_encoder = FrozenVideoEncoder(video_weights_path)

    def forward(self, rf_input, audio_input, video_input):
        token_r = self.rf_encoder(rf_input).squeeze(1)       # [B, 256]
        token_a = self.audio_encoder(audio_input).squeeze(1) # [B, 256]
        token_v = self.video_encoder(video_input).squeeze(1) # [B, 256]
        return token_r, token_a, token_v


# ── Supervised Contrastive Loss ───────────────────────────────────────────────

class AdaptiveSupConLoss(nn.Module):
    def __init__(self, temperature: float = 0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, features: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        device = features.device
        N = features.shape[0]

        z         = F.normalize(features, p=2, dim=1)
        sim       = torch.matmul(z, z.T) / self.temperature          # [N, N]
        self_mask = torch.eye(N, dtype=torch.bool, device=device)

        labels   = labels.view(-1)
        pos_mask = (labels.unsqueeze(0) == labels.unsqueeze(1)) & ~self_mask

        num_pos = pos_mask.sum(dim=1).float()
        valid   = num_pos > 0

        # logsumexp over all non-self pairs (never fill non-self with -inf here)
        sim_no_self = sim.masked_fill(self_mask, float("-inf"))
        log_prob    = sim_no_self - torch.logsumexp(sim_no_self, dim=1, keepdim=True)

        # guard: if logsumexp produced -inf (degenerate batch), skip
        if torch.isnan(log_prob).any() or torch.isinf(log_prob).all():
            return torch.tensor(0.0, device=device, requires_grad=True)

        mean_log_prob_pos = (pos_mask.float() * log_prob).sum(dim=1) / (num_pos + 1e-8)
        loss = -mean_log_prob_pos[valid].mean()

        return loss if not torch.isnan(loss) else torch.tensor(0.0, device=device, requires_grad=True)

# ── Contrastive training step ─────────────────────────────────────────────────

# ── Contrastive training step ─────────────────────────────────────────────────

def train_contrastive_step(projection_net, batch, optimizer, lr_scheduler, criterion, device) -> float:
    projection_net.train()

    rf_input    = batch["rf_input"].to(device)
    audio_input = batch["audio_input"].to(device)
    video_input = batch["video_input"].to(device)
    labels      = batch["label"].to(device)

    # ---> THE FIX: Generate the projected vectors first <---
    token_r, token_a, token_v = projection_net(rf_input, audio_input, video_input)

    # stack all three modality tokens → [3B, 256], labels → [3B]
    features = torch.cat([token_r, token_a, token_v], dim=0)
    lbls     = torch.cat([labels,  labels,  labels],  dim=0)

    loss = criterion(features, lbls)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss.item()


# ── Fusion Transformer ────────────────────────────────────────────────────────

class ModalityFusionTransformer(nn.Module):
    def __init__(
        self,
        embed_dim:   int   = EMBEDDING_DIM,
        num_heads:   int   = 4,
        num_layers:  int   = 2,
        num_classes: int   = NUM_CLASSES,
        dropout:     float = 0.1,
    ):
        super().__init__()
        # one learnable ID tag per sensor [3, 256]
        self.modality_tags = nn.Parameter(torch.randn(3, embed_dim) * 0.02)
        # prepended CLS token
        self.cls_token     = nn.Parameter(torch.randn(1, 1, embed_dim) * 0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads,
            dim_feedforward=embed_dim * 4,
            dropout=dropout, activation="gelu",
            batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers,
                                                  enable_nested_tensor=False)

        self.classifier = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, embed_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim // 2, num_classes),
        )

    def forward(self, token_r, token_a, token_v):
        B = token_r.shape[0]

        # inject modality ID tags
        x_r = token_r + self.modality_tags[0]
        x_a = token_a + self.modality_tags[1]
        x_v = token_v + self.modality_tags[2]

        # [B, 3, 256] → prepend CLS → [B, 4, 256]
        seq = torch.cat([self.cls_token.expand(B, -1, -1),
                         torch.stack([x_r, x_a, x_v], dim=1)], dim=1)

        # self-attention across all 4 tokens
        cls_out = self.transformer(seq)[:, 0, :]   # harvest CLS → [B, 256]
        return self.classifier(cls_out)             # [B, num_classes]


# ── Fusion training step ──────────────────────────────────────────────────────

import random

# ── Fusion training step ──────────────────────────────────────────────────────

def train_fusion_step(projection_net, fusion_net, batch, optimizer, scheduler, criterion, device, dropout_prob: float = 0.3) -> float:
    projection_net.eval()
    fusion_net.train()

    rf_input    = batch["rf_input"].to(device)
    audio_input = batch["audio_input"].to(device)
    video_input = batch["video_input"].to(device)
    labels      = batch["label"].to(device)

    with torch.no_grad():
        token_r, token_a, token_v = projection_net(rf_input, audio_input, video_input)

    # --- Modality Dropout ("Greying out" tokens) ---
    # Randomly zero out entire modalities with probability `dropout_prob`
    mask_r = (torch.rand(token_r.shape[0], 1, device=device) > dropout_prob).float()
    mask_a = (torch.rand(token_a.shape[0], 1, device=device) > dropout_prob).float()
    mask_v = (torch.rand(token_v.shape[0], 1, device=device) > dropout_prob).float()

    token_r = token_r * mask_r
    token_a = token_a * mask_a
    token_v = token_v * mask_v

    # Failsafe: if all three got dropped for a sample, restore the video token
    all_dropped = (mask_r + mask_a + mask_v) == 0
    if all_dropped.any():
        token_v[all_dropped.squeeze()] = projection_net.video_encoder(video_input[all_dropped.squeeze()]).squeeze(1)

    logits = fusion_net(token_r, token_a, token_v)
    loss   = criterion(logits, labels)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    return loss.item()

# ── Verification (real weights) ──────────────────────────────────────────────

if __name__ == "__main__":
    print(f"Device: {DEVICE}\n")

    B            = 8
    dummy_rf     = torch.randn(B, 2, 128,      device=DEVICE)
    dummy_audio  = torch.randn(B, 1, 128, 44,  device=DEVICE)
    dummy_video  = torch.randn(B, 3, 224, 224, device=DEVICE)
    dummy_labels = torch.tensor([1,1,0,0,1,0,1,0], dtype=torch.long, device=DEVICE)

    # Load real models
    net    = MultiModalProjectionNetwork().to(DEVICE)
    fusion = ModalityFusionTransformer().to(DEVICE)

    # Shape check — frozen backbones, no grad needed
    net.eval()
    with torch.no_grad():
        tr, ta, tv = net(dummy_rf, dummy_audio, dummy_video)

    assert tr.shape == (B, EMBEDDING_DIM)
    assert ta.shape == (B, EMBEDDING_DIM)
    assert tv.shape == (B, EMBEDDING_DIM)
    print(f"Projection shapes: RF {tr.shape}  Audio {ta.shape}  Video {tv.shape}")

    # Fusion forward pass
    logits = fusion(tr, ta, tv)
    assert logits.shape == (B, NUM_CLASSES)
    print(f"Fusion output shape: {logits.shape}")

    # Fusion training step with CE loss (works fine with dummy inputs)
    net.eval()
    fusion.train()
    opt_fuse = torch.optim.Adam(fusion.parameters(), lr=1e-4)
    ce = nn.CrossEntropyLoss()
    with torch.no_grad():
        tr, ta, tv = net(dummy_rf, dummy_audio, dummy_video)
    loss = ce(fusion(tr, ta, tv), dummy_labels)
    opt_fuse.zero_grad()
    loss.backward()
    opt_fuse.step()
    print(f"Fusion CE loss:    {loss.item():.4f}")

    print("\nShape checks passed.")
    print("NOTE: SupCon loss is only meaningful with real sensor data — run train_multimodal.py to train.")

Writing model.py


In [22]:
%%writefile train_supcon.py
import os
import torch
import wandb
from io import BytesIO
from datasets import Dataset
from torch.utils.data import DataLoader
from huggingface_hub import hf_hub_download, create_repo, HfApi

from model import MultiModalProjectionNetwork, AdaptiveSupConLoss, train_contrastive_step
from model import DEVICE

MODEL_REPO_ID = "James-ARL-2026/Drone-Multi-Modal-Model"
SUPCON_PT_PATH = "supcon_model.pt"
DATA_REPO_ID = "James-ARL-2026/Drone-Master-Snapshot-Dataset"
PARQUET_PATH = "combined_multimodal_dataset.parquet"
BATCH_SIZE = 16
NUM_EPOCHS = 4

def get_secret(key_name):
    try:
        from google.colab import userdata
        return userdata.get(key_name)
    except:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(key_name)
    except:
        pass
    return os.getenv(key_name)

# ── Custom Byte Decoding & Collate ────────────────────────────────────────────

def bytes_to_tensor(b, dtype=torch.float32):
    """Reverses the custom byte encoding from the dataset generator."""
    d = torch.load(BytesIO(b), weights_only=False)
    return torch.tensor(d["data"], dtype=dtype).reshape(d["shape"])

def multimodal_collate_fn(batch_samples):
    """Deserializes bytes and stacks them into batched PyTorch tensors."""
    batch_rf = []
    batch_audio = []
    batch_video = []
    batch_labels = []

    for sample in batch_samples:
        # Decode the bytes back into tensors using the helper function
        batch_rf.append(bytes_to_tensor(sample["rf_input"]))
        batch_audio.append(bytes_to_tensor(sample["audio_input"]))
        batch_video.append(bytes_to_tensor(sample["video_input"]))
        batch_labels.append(torch.tensor(sample["label"], dtype=torch.long))

    return {
        "rf_input": torch.stack(batch_rf, dim=0),
        "audio_input": torch.stack(batch_audio, dim=0),
        "video_input": torch.stack(batch_video, dim=0),
        "label": torch.stack(batch_labels, dim=0)
    }

# ──────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    hf_token = get_secret("HF_TOKEN")
    wandb_key = get_secret("WANDB_API_KEY")
    
    # Initialize W&B
    wandb.login(key=wandb_key)
    wandb.init(
        project="Drone-Multi-Modal", 
        name="Stage1-SupCon-Training",
        config={"batch_size": BATCH_SIZE, "epochs": NUM_EPOCHS, "stage": "SupCon"}
    )

    api = HfApi(token=hf_token)
    projection_net = MultiModalProjectionNetwork().to(DEVICE)

    # Create Repo
    create_repo(
        repo_id=MODEL_REPO_ID,
        token=hf_token,
        repo_type="model",
        private=False,
        exist_ok=True 
    )

    # Download Dataset
    parquet_file_path = hf_hub_download(
        repo_id=DATA_REPO_ID,
        filename=PARQUET_PATH,
        repo_type="dataset",
        token=hf_token,
        local_dir=".",
        local_dir_use_symlinks=False
    )

    dataset = Dataset.from_parquet(path_or_paths=parquet_file_path, keep_in_memory=True)
    
    # 90/10 Split (Removed .with_format("torch") as collate_fn handles it)
    split_dataset = dataset.train_test_split(test_size=0.1, seed=67)
    train_dataset = split_dataset["train"]
    val_dataset = split_dataset["test"]

    # Inject the byte-decoding collate function
    train_data_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, num_workers=0, drop_last=True, collate_fn=multimodal_collate_fn
    )
    val_data_loader = DataLoader(
        val_dataset, batch_size=BATCH_SIZE, num_workers=0, drop_last=False, collate_fn=multimodal_collate_fn
    )

    # Optimizer & Scheduler
    optimizer = torch.optim.AdamW(projection_net.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
    criterion = AdaptiveSupConLoss(temperature=0.07)

    # Training Loop
    for epoch in range(NUM_EPOCHS):
        projection_net.train()
        train_loss = 0.0
        
        for step, batch in enumerate(train_data_loader):
            loss = train_contrastive_step(projection_net, batch, optimizer, scheduler, criterion, DEVICE)
            train_loss += loss
            
        avg_train_loss = train_loss / len(train_data_loader)
        
        # Validation Loop
        projection_net.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch in val_data_loader:
                rf_input = batch["rf_input"].to(DEVICE)
                audio_input = batch["audio_input"].to(DEVICE)
                video_input = batch["video_input"].to(DEVICE)
                labels = batch["label"].to(DEVICE)
                
                # Generate tokens for validation loss
                token_r, token_a, token_v = projection_net(rf_input, audio_input, video_input)
                features = torch.cat([token_r, token_a, token_v], dim=0)
                lbls = torch.cat([labels, labels, labels], dim=0)
                
                v_loss = criterion(features, lbls)
                val_loss += v_loss.item()
                
        avg_val_loss = val_loss / len(val_data_loader)
        scheduler.step()
        
        print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
        wandb.log({"epoch": epoch+1, "train_loss": avg_train_loss, "val_loss": avg_val_loss})

    # Save and Push
    torch.save(projection_net.state_dict(), SUPCON_PT_PATH)
    api.upload_file(
        path_or_fileobj=SUPCON_PT_PATH,
        path_in_repo=SUPCON_PT_PATH,
        repo_id=MODEL_REPO_ID,
        repo_type="model"
    )
    print("SupCon weights successfully pushed to Hugging Face Hub!")
    wandb.finish()

Overwriting train_supcon.py


In [24]:
!python train_supcon.py


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/kaggle/working/train_supcon.py", line 2, in <module>
    import torch
  File "/usr/local/lib/python3.12/dist-packages/torch/__init__.py", line 1477, in <module>
    from .functional import *  # noqa: F403
  File "/usr/local/lib/python3.12/dist-packages/torch/functional.py", line 9, in <module>
    import torch.nn.functional as F
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/__init__.py", line 1, in <module>
    from .modules import *  # noqa: F403
  File "/usr/local/lib/python3.12/dist-packages/torch/nn

In [31]:
%%writefile train_fusion.py
import os
import torch
import wandb
from io import BytesIO
from datasets import Dataset
from torch.utils.data import DataLoader
from huggingface_hub import hf_hub_download, HfApi

from model import MultiModalProjectionNetwork, ModalityFusionTransformer, train_fusion_step
from model import DEVICE

MODEL_REPO_ID = "James-ARL-2026/Drone-Multi-Modal-Model"
SUPCON_PT_PATH = "supcon_model.pt"
FUSION_PT_PATH = "fusion_model.pt"
DATA_REPO_ID = "James-ARL-2026/Drone-Master-Snapshot-Dataset"
PARQUET_PATH = "combined_multimodal_dataset.parquet"
BATCH_SIZE = 16
NUM_EPOCHS = 8

def get_secret(key_name):
    try:
        from google.colab import userdata
        return userdata.get(key_name)
    except:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(key_name)
    except:
        pass
    return os.getenv(key_name)

# ── Custom Byte Decoding & Collate ────────────────────────────────────────────

def bytes_to_tensor(b, dtype=torch.float32):
    """Reverses the custom byte encoding from the dataset generator."""
    d = torch.load(BytesIO(b), weights_only=False)
    return torch.tensor(d["data"], dtype=dtype).reshape(d["shape"])

def multimodal_collate_fn(batch_samples):
    """Deserializes bytes and stacks them into batched PyTorch tensors."""
    batch_rf = []
    batch_audio = []
    batch_video = []
    batch_labels = []

    for sample in batch_samples:
        batch_rf.append(bytes_to_tensor(sample["rf_input"]))
        batch_audio.append(bytes_to_tensor(sample["audio_input"]))
        batch_video.append(bytes_to_tensor(sample["video_input"]))
        batch_labels.append(torch.tensor(sample["label"], dtype=torch.long))

    return {
        "rf_input": torch.stack(batch_rf, dim=0),
        "audio_input": torch.stack(batch_audio, dim=0),
        "video_input": torch.stack(batch_video, dim=0),
        "label": torch.stack(batch_labels, dim=0)
    }

# ──────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    hf_token = get_secret("HF_TOKEN")
    wandb_key = get_secret("WANDB_API_KEY")
    
    # Initialize W&B
    wandb.login(key=wandb_key)
    wandb.init(
        project="Drone-Multi-Modal", 
        name="Stage2-Fusion-Training",
        config={"batch_size": BATCH_SIZE, "epochs": NUM_EPOCHS, "stage": "Fusion"}
    )

    api = HfApi(token=hf_token)

    print("Downloading Stage 1 SupCon weights...")
    supcon_weights_path = hf_hub_download(
        repo_id=MODEL_REPO_ID, filename=SUPCON_PT_PATH, token=hf_token
    )
    
    # 1. Initialize Projection Network and completely freeze it
    projection_net = MultiModalProjectionNetwork().to(DEVICE)
    projection_net.load_state_dict(torch.load(supcon_weights_path, map_location=DEVICE))
    projection_net.eval()  # CRITICAL: Freezes dropout/batchnorm layers

    # 2. Initialize fresh Fusion Network
    fusion_net = ModalityFusionTransformer().to(DEVICE)

    # 3. Setup Dataset (Must match the same 90/10 seed as SupCon)
    print("Preparing dataset...")
    parquet_file_path = hf_hub_download(
        repo_id=DATA_REPO_ID, filename=PARQUET_PATH, repo_type="dataset", token=hf_token
    )
    dataset = Dataset.from_parquet(path_or_paths=parquet_file_path, keep_in_memory=True)
    split_dataset = dataset.train_test_split(test_size=0.1, seed=67)
    
    train_dataset = split_dataset["train"]
    val_dataset = split_dataset["test"]

    train_data_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, num_workers=0, drop_last=True, collate_fn=multimodal_collate_fn
    )
    val_data_loader = DataLoader(
        val_dataset, batch_size=BATCH_SIZE, num_workers=0, drop_last=False, collate_fn=multimodal_collate_fn
    )

    # 4. Optimizer & Scheduler
    optimizer = torch.optim.AdamW(fusion_net.parameters(), lr=1e-4, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
    criterion = torch.nn.CrossEntropyLoss()

    print("Starting Fusion Training...")
    for epoch in range(NUM_EPOCHS):
        fusion_net.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        for step, batch in enumerate(train_data_loader):
            labels = batch["label"].to(DEVICE)
            
            with torch.no_grad():
                tr, ta, tv = projection_net(
                    batch["rf_input"].to(DEVICE), 
                    batch["audio_input"].to(DEVICE), 
                    batch["video_input"].to(DEVICE)
                )
            
            loss = train_fusion_step(projection_net, fusion_net, batch, optimizer, scheduler, criterion, DEVICE)
            train_loss += loss
            
            logits = fusion_net(tr, ta, tv)
            preds = torch.argmax(logits, dim=1)
            train_correct += (preds == labels).sum().item()
            train_total += labels.size(0)
            
        avg_train_loss = train_loss / len(train_data_loader)
        train_acc = train_correct / train_total
        
        # Validation Loop
        fusion_net.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for batch in val_data_loader:
                rf_input = batch["rf_input"].to(DEVICE)
                audio_input = batch["audio_input"].to(DEVICE)
                video_input = batch["video_input"].to(DEVICE)
                labels = batch["label"].to(DEVICE)
                
                tr, ta, tv = projection_net(rf_input, audio_input, video_input)
                logits = fusion_net(tr, ta, tv)
                
                v_loss = criterion(logits, labels)
                val_loss += v_loss.item()
                
                preds = torch.argmax(logits, dim=1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)
                
        avg_val_loss = val_loss / len(val_data_loader)
        val_acc = val_correct / val_total
        scheduler.step()
        
        print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Train Loss: {avg_train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {avg_val_loss:.4f} Acc: {val_acc:.4f}")
        wandb.log({
            "epoch": epoch+1, 
            "train_loss": avg_train_loss, 
            "train_acc": train_acc, 
            "val_loss": avg_val_loss, 
            "val_acc": val_acc
        })

    # Save and push weights
    print("Pushing Fusion weights to Hugging Face...")
    torch.save(fusion_net.state_dict(), FUSION_PT_PATH)
    api.upload_file(
        path_or_fileobj=FUSION_PT_PATH,
        path_in_repo=FUSION_PT_PATH,
        repo_id=MODEL_REPO_ID,
        repo_type="model"
    )
    print("Done!")
    wandb.finish()

Overwriting train_fusion.py


In [32]:
!python train_fusion.py


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/kaggle/working/train_fusion.py", line 2, in <module>
    import torch
  File "/usr/local/lib/python3.12/dist-packages/torch/__init__.py", line 1477, in <module>
    from .functional import *  # noqa: F403
  File "/usr/local/lib/python3.12/dist-packages/torch/functional.py", line 9, in <module>
    import torch.nn.functional as F
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/__init__.py", line 1, in <module>
    from .modules import *  # noqa: F403
  File "/usr/local/lib/python3.12/dist-packages/torch/nn

In [37]:
import os
import torch
from io import BytesIO
from datasets import Dataset
from torch.utils.data import DataLoader
from huggingface_hub import hf_hub_download

# Import your network architectures and constants
from model import MultiModalProjectionNetwork, ModalityFusionTransformer, DEVICE

MODEL_REPO_ID = "James-ARL-2026/Drone-Multi-Modal-Model"
DATA_REPO_ID = "James-ARL-2026/Drone-Master-Snapshot-Dataset"
PARQUET_PATH = "combined_multimodal_dataset.parquet"

# Ensure these match the filenames you pushed to Hugging Face
SUPCON_PT_PATH = "supcon_model.pt"
FUSION_PT_PATH = "fusion_model.pt"

def get_secret(key_name):
    try:
        from google.colab import userdata
        return userdata.get(key_name)
    except:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(key_name)
    except:
        pass
    return os.getenv(key_name)

# ── Custom Byte Decoding & Collate ────────────────────────────────────────────

def bytes_to_tensor(b, dtype=torch.float32):
    """Reverses the custom byte encoding from the dataset generator."""
    d = torch.load(BytesIO(b), weights_only=False)
    return torch.tensor(d["data"], dtype=dtype).reshape(d["shape"])

def multimodal_collate_fn(batch_samples):
    """Deserializes bytes and stacks them into batched PyTorch tensors."""
    batch_rf = []
    batch_audio = []
    batch_video = []
    batch_labels = []

    for sample in batch_samples:
        batch_rf.append(bytes_to_tensor(sample["rf_input"]))
        batch_audio.append(bytes_to_tensor(sample["audio_input"]))
        batch_video.append(bytes_to_tensor(sample["video_input"]))
        batch_labels.append(torch.tensor(sample["label"], dtype=torch.long))

    return {
        "rf_input": torch.stack(batch_rf, dim=0),
        "audio_input": torch.stack(batch_audio, dim=0),
        "video_input": torch.stack(batch_video, dim=0),
        "label": torch.stack(batch_labels, dim=0)
    }

# ──────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    hf_token = get_secret("HF_TOKEN")

    print(f"Using Device: {DEVICE}")

    # 1. Download and Prepare the Test Dataset
    print("Downloading dataset to extract the test split...")
    parquet_file_path = hf_hub_download(
        repo_id=DATA_REPO_ID, 
        filename=PARQUET_PATH, 
        repo_type="dataset", 
        token=hf_token
    )
    
    # We use the exact same seed (67) so we reliably extract the 10% unseen test set
    dataset = Dataset.from_parquet(path_or_paths=parquet_file_path, keep_in_memory=True)
    split_dataset = dataset.train_test_split(test_size=0.1, seed=67)
    
    # Removed .with_format("torch") because collate_fn handles it
    test_dataset = split_dataset["test"]
    
    # Injected the collate_fn
    test_data_loader = DataLoader(
        test_dataset, 
        batch_size=32, 
        shuffle=True, # Shuffle to see different random samples on each run
        num_workers=0, 
        drop_last=True,
        collate_fn=multimodal_collate_fn
    )

    # 2. Download the trained weights
    print("Downloading Stage 1 (SupCon) and Stage 2 (Fusion) weights...")
    supcon_weights_path = hf_hub_download(
        repo_id=MODEL_REPO_ID, 
        filename=SUPCON_PT_PATH, 
        token=hf_token
    )
    fusion_weights_path = hf_hub_download(
        repo_id=MODEL_REPO_ID, 
        filename=FUSION_PT_PATH, 
        token=hf_token
    )

    # 3. Initialize Models and Load Weights
    print("Initializing networks...")
    
    projection_net = MultiModalProjectionNetwork().to(DEVICE)
    projection_net.load_state_dict(torch.load(supcon_weights_path, map_location=DEVICE))
    projection_net.eval()

    fusion_net = ModalityFusionTransformer().to(DEVICE)
    fusion_net.load_state_dict(torch.load(fusion_weights_path, map_location=DEVICE))
    fusion_net.eval()

    # 4. Run Inference on a Single Batch
    print("\n" + "="*50)
    print(" RUNNING INFERENCE ON TEST BATCH")
    print("="*50)

    # Grab one batch from the test loader
    batch = next(iter(test_data_loader))
    
    rf_input = batch["rf_input"].to(DEVICE)
    audio_input = batch["audio_input"].to(DEVICE)
    video_input = batch["video_input"].to(DEVICE)
    true_labels = batch["label"].to(DEVICE)

    with torch.no_grad():
        # Stage 1: Project modalities into shared embedding space
        token_r, token_a, token_v = projection_net(rf_input, audio_input, video_input)
        
        # Stage 2: Fuse tokens and get classification logits
        logits = fusion_net(token_r, token_a, token_v)
        
        # Calculate probabilities and final predicted classes
        probabilities = torch.softmax(logits, dim=1)
        predictions = torch.argmax(probabilities, dim=1)

    # 5. Display Results
    correct_count = 0
    total_samples = len(true_labels)

    for i in range(total_samples):
        true_lbl = true_labels[i].item()
        pred_lbl = predictions[i].item()
        conf = probabilities[i][pred_lbl].item() * 100
        
        match = "✅" if true_lbl == pred_lbl else "❌"
        if true_lbl == pred_lbl:
            correct_count += 1
            
        print(f"Sample {i+1} {match} | True Label: {true_lbl} | Predicted: {pred_lbl} | Confidence: {conf:.2f}%")

    print("-" * 50)
    print(f"Batch Accuracy: {correct_count}/{total_samples} ({(correct_count/total_samples)*100:.1f}%)")
    print("=" * 50)

Using Device: cuda
Initializing networks...
Loaded RF model from ./ARL/sensor_client.pt

 RUNNING INFERENCE ON TEST BATCH
Sample 1 ✅ | True Label: 1 | Predicted: 1 | Confidence: 100.00%
Sample 2 ✅ | True Label: 1 | Predicted: 1 | Confidence: 100.00%
Sample 3 ✅ | True Label: 0 | Predicted: 0 | Confidence: 100.00%
Sample 4 ✅ | True Label: 0 | Predicted: 0 | Confidence: 100.00%
Sample 5 ✅ | True Label: 0 | Predicted: 0 | Confidence: 100.00%
Sample 6 ✅ | True Label: 1 | Predicted: 1 | Confidence: 100.00%
Sample 7 ✅ | True Label: 1 | Predicted: 1 | Confidence: 100.00%
Sample 8 ✅ | True Label: 0 | Predicted: 0 | Confidence: 100.00%
Sample 9 ✅ | True Label: 1 | Predicted: 1 | Confidence: 100.00%
Sample 10 ✅ | True Label: 0 | Predicted: 0 | Confidence: 100.00%
Sample 11 ✅ | True Label: 1 | Predicted: 1 | Confidence: 100.00%
Sample 12 ✅ | True Label: 1 | Predicted: 1 | Confidence: 100.00%
Sample 13 ✅ | True Label: 0 | Predicted: 0 | Confidence: 100.00%
Sample 14 ✅ | True Label: 1 | Predicted: 1